In [1]:
!pip install deepface opencv-python-headless matplotlib pandas tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.3 MB/s eta 0:00:00


# New Section

In [5]:
import os

weights_path = '/root/.deepface/weights/gender_model_weights.h5'
if os.path.exists(weights_path):
    os.remove(weights_path)
    print(f"✔ Successfully deleted corrupted DeepFace weights file: {weights_path}")
else:
    print(f"ℹ️ DeepFace weights file not found at: {weights_path}. It might have already been removed or never downloaded.")


ℹ️ DeepFace weights file not found at: /root/.deepface/weights/gender_model_weights.h5. It might have already been removed or never downloaded.


In [4]:
import cv2
import numpy as np
from deepface import DeepFace
from google.colab.patches import cv2_imshow
from IPython.display import clear_output
import time
import pandas as pd
import matplotlib.pyplot as plt

# =============================
# CONFIG
# =============================
video_path = '/content/sample27.mp4'  # change to your video file
frame_skip = 5                        # analyze every 5th frame
movement_threshold = 15               # threshold in pixels

male_positions = []   # stores centroid positions per frame
prev_men_centroids = []

# =============================
# OPEN VIDEO
# =============================
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("❌ VIDEO NOT FOUND!")
else:
    print("✔ Video opened")

frame_num = 0

# =============================
# PROCESS FRAMES
# =============================
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_num += 1

        if frame_num % frame_skip != 0:
            continue

        # resize for faster detection
        small_frame = cv2.resize(frame, (0,0), fx=0.5, fy=0.5)

        try:
            results = DeepFace.analyze(
                small_frame,
                actions=['gender'],
                detector_backend='opencv',
                enforce_detection=False
            )
        except Exception as deepface_e:
            print(f"⚠️ DeepFace analysis failed for frame {frame_num}: {deepface_e}")
            results = []

        if isinstance(results, dict):
            results = [results]

        current_men_centroids = []
        frame_danger = False

        # DRAW & COLLECT
        for face in results:
            x = int(face['region']['x'] * 2)
            y = int(face['region']['y'] * 2)
            w = int(face['region']['w'] * 2)
            h = int(face['region']['h'] * 2)
            gender = face['gender']
            label = "Male" if gender == "Man" else "Female" # Changed to detect Male
            color = (255, 0, 0) if label == "Male" else (0, 0, 255) # Red for Male, Blue for Female

            cv2.rectangle(frame, (x,y), (x+w, y+h), color, 2)
            cv2.putText(frame, label, (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

            if label == "Male": # Changed to Male
                cx, cy = x + w//2, y + h//2
                current_men_centroids.append((cx, cy))

        male_positions.append(current_men_centroids) # Changed to male_positions

        # MOVEMENT ALERT
        if prev_men_centroids and current_men_centroids: # Changed to men_centroids
            for i, (cx, cy) in enumerate(current_men_centroids): # Changed to men_centroids
                if i < len(prev_men_centroids): # Changed to men_centroids
                    px, py = prev_men_centroids[i]
                    dist = np.sqrt((cx - px)**2 + (cy - py)**2)
                    if dist < movement_threshold:
                        print(f"⚠ Frame {frame_num}: Male movement {dist:.2f} < threshold") # Updated message
                        frame_danger = True

        if frame_danger:
            # Draw a red border around the entire entire frame
            cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 0, 255), 5)

        prev_men_centroids = current_men_centroids # Changed to men_centroids

        # SHOW FRAME
        clear_output(wait=True)
        cv2_imshow(frame)
        time.sleep(0.1)
except Exception as e:
    print(f"❌ An unexpected error occurred during video processing: {e}")
    import traceback
    traceback.print_exc()
finally:
    cap.release()
    print("✔ Processing complete.")

❌ VIDEO NOT FOUND!
✔ Processing complete.


# New Section

# New Section